# Day 4 — tidy data와 메타데이터: pandas로 표 건강검진하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-1/day-04.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-1/day-04.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 넓은 표를 tidy한 긴 표로 바꾸기

In [ ]:
import pandas as pd

# 넓은(wide) 표: 유전자마다 열이 하나씩 — 사람이 보기엔 편하다 (가상 데이터)
wide = pd.DataFrame({
    "sample_id": ["S1", "S2", "S3", "S4"],
    "group":     ["control", "control", "flight", "flight"],
    "Gfap":      [9.8, 10.4, 13.5, 14.1],
    "Sod2":      [8.1, 7.9, 8.6, 8.2],
})
print(wide)

# 긴(long, tidy) 표: 한 행 = 한 번의 측정
tidy = wide.melt(id_vars=["sample_id", "group"], var_name="gene", value_name="expression")
print(tidy)

# tidy 형태면 "군 × 유전자" 요약이 한 줄이다
print(tidy.groupby(["gene", "group"])["expression"].mean())

### 예시 2 — 문제가 숨어 있는 CSV 점검하기

일부러 문제 다섯 가지를 심어 둔 임상 표를 만들고 읽어 봅니다. 실행하기 전에 `csv_text`를 눈으로 보고 문제를 몇 개 찾을 수 있는지 먼저 세어 보세요.

In [ ]:
import pandas as pd

# 일부러 문제를 심어 둔 가상 임상 표를 CSV 파일로 만든다
csv_text = """participant_id,group,age,sex,tremor_score
P001,PD,67,F,2.1
P002,Control,64,M,0.3
P003,PD,unknown,M,3.4
P004,control,59,F,
P005,PD,71,F,2.8
P003,PD,unknown,M,3.4
P006,PD ,58,M,1.9
"""
with open("../data/toy_clinical.csv", "w") as f:
    f.write(csv_text)

df = pd.read_csv("../data/toy_clinical.csv")

print("크기(행, 열):", df.shape)
print()
print(df.dtypes)
print()
print("열별 결측 개수:")
print(df.isna().sum())
print()
print("중복된 ID:", df.loc[df["participant_id"].duplicated(), "participant_id"].tolist())
print()
print(df["group"].value_counts())

### 예시 3 — 고치고, 다시 검진하기

In [ ]:
import pandas as pd

df = pd.read_csv("../data/toy_clinical.csv")

# 1) 완전히 똑같은 행은 한 번만 남긴다
df = df.drop_duplicates()

# 2) 군 이름 표기를 통일한다: 앞뒤 공백 제거 + 대소문자 통일
df["group"] = df["group"].str.strip().replace({"control": "Control"})

# 3) age를 숫자로 바꾼다. 숫자가 아닌 값("unknown")은 결측(NaN)이 된다
df["age"] = pd.to_numeric(df["age"], errors="coerce")

print(df)
print()
print(df.dtypes)
print()
print(df["group"].value_counts())
print()
print("열별 결측 개수:")
print(df.isna().sum())

## 직접 해보기

3. **QC 함수를 만듭니다.** 매번 네 줄을 치는 대신, 프로토콜 하나로 묶습니다.

In [ ]:
import pandas as pd

def qc_report(df, id_col):
    """표 하나를 받아 기본 QC 결과를 출력한다."""
    print(f"행 {df.shape[0]}개, 열 {df.shape[1]}개")
    print("- 완전히 중복된 행:", int(df.duplicated().sum()))
    print("- 중복된 ID:", sorted(df.loc[df[id_col].duplicated(), id_col].unique().tolist()))
    missing = df.isna().sum()
    print("- 결측이 있는 열:", missing[missing > 0].to_dict())
    text_cols = df.select_dtypes(include="object").columns.tolist()
    print("- 글자(object)로 읽힌 열:", text_cols)

metadata = pd.read_csv("../data/toy_clinical.csv")
qc_report(metadata, id_col="participant_id")

5. **두 표의 ID가 맞는지 확인합니다.** 측정값 표와 메타데이터 표를 잇기 전에 반드시 하는 점검입니다.

In [ ]:
import pandas as pd

expression = pd.DataFrame({
    "sample_id": ["S1", "S2", "S3", "S5"],
    "Gfap": [9.8, 10.4, 13.5, 12.0],
})
metadata = pd.DataFrame({
    "sample_id": ["S1", "S2", "S3", "S4"],
    "group": ["control", "control", "flight", "flight"],
})

only_in_expression = set(expression["sample_id"]) - set(metadata["sample_id"])
only_in_metadata = set(metadata["sample_id"]) - set(expression["sample_id"])
print("발현 표에만 있는 ID:", only_in_expression)
print("메타데이터에만 있는 ID:", only_in_metadata)

merged = expression.merge(metadata, on="sample_id", how="left")
print(merged)

7. **저장, 일지, commit.**

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# git add notebooks/01_setup_and_metadata.ipynb learning-log.md
# git commit -m "Day 4: 결측치·중복 ID 점검 함수 작성"